# 02 — Error analysis (false positives / false negatives)

Loads the cached dev candidates + stage-1 features for a given cache tag, retrains the stage-1 OOF
model (not persisted to disk, so this re-runs `model.train_oof` — ~20-25 min at `FRAC=0.1`), then shows
false positives and false negatives side by side with their raw (pre-normalization) records. Filter by
editing `COUNTRY` / `ERROR_TYPE` in the parameters cell and re-running the last cell.

No pipeline logic here beyond calling `src/` functions directly (same helpers `run_pipeline.py` uses).


In [ ]:
import pathlib, sys

NB_DIR = pathlib.Path.cwd().resolve()
_d = NB_DIR
while not (_d / "CLAUDE.md").exists():
    if _d == _d.parent:
        raise RuntimeError("could not find student_resource/ (looked for a CLAUDE.md marker)")
    _d = _d.parent
REPO_ROOT = _d
PKG_DIR = REPO_ROOT / "code" / "business_entity_resolution"
if str(PKG_DIR) not in sys.path:
    sys.path.insert(0, str(PKG_DIR))
DATA_DIR = REPO_ROOT / "dataset"
OUT_DIR = REPO_ROOT / "output"
print("repo root:", REPO_ROOT)


## Parameters

In [ ]:
import os

TAG = "v1"            # cache tag to read/write (code/business_entity_resolution/cache/<TAG>_*)
FRAC = 0.1             # must match a cached cands_train_<FRAC> / feats1_train_<FRAC>, or it will be built
COUNTRY = None         # e.g. "US", "India" or None for all
ERROR_TYPE = None      # one of: blocking_miss, native_script, generic_word, legal_moved, addr_norm, number, other, None for all
N_EXAMPLES = 15

os.environ["ER_CACHE_TAG"] = TAG
os.environ.setdefault("ER_JOBS", "4")
os.environ.setdefault("ER_LGB_JOBS", str(os.cpu_count() or 4))


In [ ]:
import numpy as np
import pandas as pd

from src import blocking, config, decide, features, io_utils, model
from src import run_pipeline as rp

print("cache tag:", config.CACHE_TAG, "| cache dir:", config.CACHE_DIR)


## Load normalized train frames, candidates and stage-1 features (cached)

In [ ]:
table = rp.get_translit(DATA_DIR)
s1n, pooln = rp.load_split_norm(DATA_DIR, "train", table)
gt = io_utils.load_ground_truth(DATA_DIR)
pos_all = rp.gt_rid_pairs(gt, s1n, pooln)

rids = rp.sample_s1(s1n, FRAC)
pos = pos_all[np.isin(pos_all["s1_rid"].to_numpy(), rids)]

cands = rp.cached_frame(f"cands_train_{FRAC}", lambda: blocking.generate_candidates(s1n, pooln, rids))
generic = rp.get_generic_vocab(s1n, pooln, pos_all)
feats = rp.cached_frame(f"feats1_train_{FRAC}", lambda: features.build_features(cands, s1n, pooln, generic))
print(f"entities={len(rids):,} positives={len(pos):,} candidate pairs={len(feats):,}")


## Blocking misses (true pairs blocking never proposed)

In [ ]:
have = rp.pair_key(feats["s1_rid"], feats["pool_rid"])
miss_mask = ~np.isin(rp.pair_key(pos["s1_rid"], pos["pool_rid"]), have)
blocking_misses = pos[miss_mask]
print(f"blocking misses: {len(blocking_misses):,} / {len(pos):,} positive pairs "
      f"({len(blocking_misses) / max(len(pos), 1):.1%})")


## OOF stage-1 scores (retrains; not cached to disk)

In [ ]:
y = np.isin(rp.pair_key(feats["s1_rid"], feats["pool_rid"]), rp.pair_key(pos["s1_rid"], pos["pool_rid"])).astype(np.int8)
groups = feats["s1_rid"].to_numpy()
cols1 = model.feature_cols(feats)
p1, _ = model.train_oof(feats[cols1], y, groups)
feats["p1"] = p1
feats["y"] = y

dec_path = rp.cache_path("decoder", "json")
T_MATCH = __import__("json").loads(dec_path.read_text())["t_match"] if dec_path.exists() else config.T_MATCH
print(f"using t_match={T_MATCH}")


## Classify errors and join raw (pre-normalization) records

In [ ]:
NOISE_COLS = {"generic_word": "f_nz_name_generic_only", "legal_moved": "f_nz_legal_moved",
              "addr_norm": "f_nz_addr_norm_only", "number": "f_num1_eq"}


def classify(row):
    """Best-guess error bucket from noise/record features, first match wins."""
    if row.get("f_script_a") or row.get("f_script_b"):
        return "native_script"
    if row.get("f_nz_name_generic_only"):
        return "generic_word"
    if row.get("f_nz_legal_moved"):
        return "legal_moved"
    if row.get("f_nz_addr_norm_only"):
        return "addr_norm"
    if row.get("f_num1_eq") == 0.0:
        return "number"
    return "other"


fp = feats[(feats["p1"] >= T_MATCH) & (feats["y"] == 0)].copy()
fp["kind"], fp["error_type"] = "false_positive", fp.apply(classify, axis=1)

fn_scored = feats[(feats["p1"] < T_MATCH) & (feats["y"] == 1)].copy()
fn_scored["kind"], fn_scored["error_type"] = "false_negative_scored_low", fn_scored.apply(classify, axis=1)

fn_blocked = blocking_misses.rename(columns={"s1_id": "s1_rid_id"}).copy()  # already s1_rid/pool_rid cols
fn_blocked["kind"], fn_blocked["error_type"], fn_blocked["p1"] = "false_negative_blocking_miss", "blocking_miss", np.nan

errs = pd.concat([fp, fn_scored, fn_blocked], ignore_index=True, sort=False)
errs["country"] = s1n["country"].to_numpy()[errs["s1_rid"].to_numpy()]

raw_s1 = io_utils.load_tsv(DATA_DIR / "train" / "train_source1.tsv").set_index("entity_id")
raw_pool = pd.concat([io_utils.load_tsv(DATA_DIR / "train" / f"train_source{k}.tsv") for k in (2, 3)],
                     ignore_index=True).set_index("entity_id")
s1_ids = s1n["entity_id"].to_numpy()
pool_ids = pooln["entity_id"].to_numpy()
print(f"false positives={len(fp):,} false negatives (scored)={len(fn_scored):,} "
      f"false negatives (blocking miss)={len(fn_blocked):,}")


## Filtered view (edit COUNTRY / ERROR_TYPE above and re-run)

In [ ]:
view = errs
if COUNTRY is not None:
    view = view[view["country"] == COUNTRY]
if ERROR_TYPE is not None:
    view = view[view["error_type"] == ERROR_TYPE]

rows = []
for _, r in view.head(N_EXAMPLES).iterrows():
    s1_id = s1_ids[int(r["s1_rid"])]
    pool_id = pool_ids[int(r["pool_rid"])]
    rows.append({
        "kind": r["kind"], "error_type": r["error_type"], "country": r["country"],
        "p1": r.get("p1", float("nan")),
        "s1_name": raw_s1.loc[s1_id, "business_name"] if s1_id in raw_s1.index else "",
        "s1_addr": raw_s1.loc[s1_id, "business_address"] if s1_id in raw_s1.index else "",
        "pool_name": raw_pool.loc[pool_id, "business_name"] if pool_id in raw_pool.index else "",
        "pool_addr": raw_pool.loc[pool_id, "business_address"] if pool_id in raw_pool.index else "",
    })
view_df = pd.DataFrame(rows)
print(f"showing {len(view_df)} of {len(view):,} matching rows "
      f"(country={COUNTRY}, error_type={ERROR_TYPE})")
display(view_df)
